In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector as select
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler


adult_df = pd.read_csv('adult.csv')
compas_df = pd.read_csv('compas.csv')
german_df = pd.read_csv('german.csv')



2. Compute the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), and the consistency from the original labels of the dataset before mitigation and training any classifiers.

In [65]:

#preprocessing
def clean(df):
    df = df.copy()
    text = df.select_dtypes(exclude="number").columns
    df[text] = df[text].apply(lambda s: s.astype(str).str.strip().str.lower())
    return df


adult_df, compas_df, german_df = clean(adult_df), clean(compas_df), clean(german_df)
adult_df["income"] = adult_df["income"].str.rstrip(".")      # ">50k." -> ">50k"

# difference between genders
adult_y = (adult_df["income"] == ">50k").astype(int)
adult_unpriv = adult_df["sex"] == "female"
adult_priv = adult_df["sex"] == "male"

#no recidivism = prefered outcome, keep only 2 races for easier comparison
compas_df = compas_df[compas_df["race"].isin(["african-american", "caucasian"])].reset_index(drop=True)
compas_y = (compas_df["two_year_recid"].astype(int) == 0).astype(int)
compas_unpriv = compas_df["race"] == "african-american"
compas_priv = compas_df["race"] == "caucasian"

#difference between genders
german_y = (german_df["target"] == "good").astype(int)
german_unpriv = german_df["status_and_sex"].str.startswith("female")
german_priv = german_df["status_and_sex"].str.startswith("male")

In [66]:
#demographic parity difference
def dp_diff(y, unpriv, priv):
    """P(+ | unprivileged) - P(+ | privileged)."""
    y = np.asarray(y, dtype=float)
    return y[np.asarray(unpriv)].mean() - y[np.asarray(priv)].mean()

#disparate impact ratio
def dir_ratio(y, unpriv, priv):
    """P(+ | unprivileged) / P(+ | privileged)."""
    y = np.asarray(y, dtype=float)
    return y[np.asarray(unpriv)].mean() / y[np.asarray(priv)].mean()

#consistency
def consistency(X, y, k=5):
    """1 - mean |y_i - mean label of i's k nearest neighbours|. X must be numeric."""
    y = np.asarray(y, dtype=float)
    idx = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)
    neighbors = np.vstack([row[row != i][:k] for i, row in enumerate(idx)])  # drop the row itself
    return 1 - np.abs(y - y[neighbors].mean(axis=1)).mean()

In [67]:
print(f"Adult: DP_Diff={dp_diff(adult_y, adult_unpriv, adult_priv):.4f}, DIR={dir_ratio(adult_y, adult_unpriv, adult_priv):.4f}, ")
print(f"COMPAS: DP_Diff={dp_diff(compas_y, compas_unpriv, compas_priv):.4f}, DIR={dir_ratio(compas_y, compas_unpriv, compas_priv):.4f}")
print(f"German: DP_Diff={dp_diff(german_y, german_unpriv, german_priv):.4f}, DIR={dir_ratio(german_y, german_unpriv, german_priv):.4f}")

Adult: DP_Diff=-0.1963, DIR=0.3580, 
COMPAS: DP_Diff=-0.1207, DIR=0.8009
German: DP_Diff=-0.0748, DIR=0.8966


3. Perform a train/test split with stratification.

In [ ]:

#stratified train test split by both label and group
def split(df, y, unpriv, priv, test_size=0.5, seed=69):

    strat = y.astype(int) * 2 + unpriv.astype(int)
    parts = train_test_split(df, y, unpriv, priv, test_size=test_size,
                             stratify=strat, random_state=seed)
    names = ["df", "y", "unpriv", "priv"]
    return {f"{n}_{s}": parts[2 * i + j]
            for i, n in enumerate(names) for j, s in enumerate(["train", "test"])}

#new data splits
data = {
    "Adult":  split(adult_df,  adult_y,  adult_unpriv,  adult_priv),
    "COMPAS": split(compas_df, compas_y, compas_unpriv, compas_priv),
    "German": split(german_df, german_y, german_unpriv, german_priv),
}

4. Follow the appropriate procedure for the selected method: pre-processing methods modify training data, in-processing methods are used during model training, and post-processing methods modify predictions after training.

In [ ]:

#standardize data
def make_prep():
    return ColumnTransformer([
        ("num", StandardScaler(), select(dtype_include="number")),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), select(dtype_exclude="number")),
    ])

#method from paper calculating probability, assigning promotions and demotions and swapping labels.
def massage(X, y, unpriv, priv):
    y_new = y.copy()
    n_s, n_p = unpriv.sum(), priv.sum()
    M = int(round((n_s * y[priv].sum() - n_p * y[unpriv].sum()) / (n_s + n_p)))

    log_p = GaussianNB().fit(X, y).predict_log_proba(X)
    score = log_p[:, 1] - log_p[:, 0]

    promo = np.where(unpriv & (y == 0))[0]
    promo = promo[np.argsort(-score[promo])][:M]
    demo = np.where(priv & (y == 1))[0]
    demo = demo[np.argsort(score[demo])][:M]

    y_new[promo], y_new[demo] = 1, 0
    return y_new, M

In [70]:
#applying CND to adult dataset
d = data["Adult"]
feats = [c for c in adult_df.columns if c not in ("income", "sex")]

prep_adult = make_prep()
X_train = prep_adult.fit_transform(d["df_train"][feats])
y_train = d["y_train"].to_numpy()
unpriv, priv = d["unpriv_train"].to_numpy(), d["priv_train"].to_numpy()

adult_y_mass, M = massage(X_train, y_train, unpriv, priv)

print("Adult flips:", M)
print("DP_Diff before:", dp_diff(y_train, unpriv, priv))
print("DP_Diff after: ", dp_diff(adult_y_mass, unpriv, priv))

Adult flips: 708
DP_Diff before: -0.1963586748327391
DP_Diff after:  0.00010158587308667366


In [71]:
#applying CND to compas dataset
d = data["COMPAS"]
feats = ["sex", "age", "juv_fel_count", "juv_misd_count",
         "juv_other_count", "priors_count", "c_charge_degree"]

prep_compas = make_prep()
X_train = prep_compas.fit_transform(d["df_train"][feats])
y_train = d["y_train"].to_numpy()
unpriv, priv = d["unpriv_train"].to_numpy(), d["priv_train"].to_numpy()

compas_y_mass, M = massage(X_train, y_train, unpriv, priv)

print("COMPAS flips:", M)
print("DP_Diff before:", dp_diff(y_train, unpriv, priv))
print("DP_Diff after: ", dp_diff(compas_y_mass, unpriv, priv))

COMPAS flips: 89
DP_Diff before: -0.12042623228442301
DP_Diff after:  0.0002685782025635408


In [72]:
#applying CND to german dataset
d = data["German"]
feats = [c for c in german_df.columns if c not in ("target", "status_and_sex")]

prep_german = make_prep()
X_train = prep_german.fit_transform(d["df_train"][feats])
y_train = d["y_train"].to_numpy()
unpriv, priv = d["unpriv_train"].to_numpy(), d["priv_train"].to_numpy()

german_y_mass, M = massage(X_train, y_train, unpriv, priv)

print("German flips:", M)
print("DP_Diff before:", dp_diff(y_train, unpriv, priv))
print("DP_Diff after: ", dp_diff(german_y_mass, unpriv, priv))

German flips: 8
DP_Diff before: -0.07012622720897621
DP_Diff after:  0.004675081813931725


5. On the held-out test data, report the demographic parity difference (DP_Diff), the disparate impact ratio (DIR), the consistency, and at least one predictive-performance measure from the final predictions after applying the complete mitigation pipeline.

In [ ]:

#calculate consistency of predictions
def consistency(X, y, k=5):
    idx = NearestNeighbors(n_neighbors=k + 1).fit(X).kneighbors(X, return_distance=False)
    neighbors = np.vstack([row[row != i][:k] for i, row in enumerate(idx)]) 
    return 1 - np.abs(y - y[neighbors].mean(axis=1)).mean()

#evaluate test set with classifier, naive bayes just like in paper
def evaluate_test(d, feats, prep, y_mass):
    X_train = prep.transform(d["df_train"][feats])
    X_test = prep.transform(d["df_test"][feats])

    pred = GaussianNB().fit(X_train, y_mass).predict(X_test)

    y_test = d["y_test"].to_numpy()
    unpriv, priv = d["unpriv_test"].to_numpy(), d["priv_test"].to_numpy()
    return dict(
        DP_Diff=dp_diff(pred, unpriv, priv),
        DIR=dir_ratio(pred, unpriv, priv),
        Consistency=consistency(X_test, pred),
        Accuracy=accuracy_score(y_test, pred),
    )

In [ ]:
#selecting columns for model and consistency evaluation
features_adult  = [c for c in adult_df.columns if c not in ("income", "sex")]
features_compas = ["sex", "age", "juv_fel_count", "juv_misd_count",
                   "juv_other_count", "priors_count", "c_charge_degree"]
features_german = [c for c in german_df.columns if c not in ("target", "status_and_sex")]

In [ ]:
#scores after applying CND to adult dataset
after_adult = evaluate_test(data["Adult"], features_adult, prep_adult, adult_y_mass)

for metric, value in after_adult.items():
    print(f"{metric}: {value:.4f}")

DP_Diff: 0.0056
DIR: 1.0073
Consistency: 0.9252
Accuracy: 0.4373


In [ ]:
#scores after applying CND to compas dataset
after_compas = evaluate_test(data["COMPAS"], features_compas, prep_compas, compas_y_mass)
for metric, value in after_compas.items():
    print(f"{metric}: {value:.4f}")

DP_Diff: -0.1321
DIR: 0.8511
Consistency: 0.9928
Accuracy: 0.6124


In [ ]:
#scores after applying CND to german dataset
after_german = evaluate_test(data["German"], features_german, prep_german, german_y_mass)

for metric, value in after_german.items():
    print(f"{metric}: {value:.4f}")
    

DP_Diff: -0.0555
DIR: 0.6638
Consistency: 0.8000
Accuracy: 0.3920


6. Compare the values of the fairness measures before and after the bias mitigation.

In [ ]:
# calculating fairness measures on the true labels
def true_label_measures(d, feats, prep):
    X_test = prep.transform(d["df_test"][feats])
    y_test = d["y_test"].to_numpy()
    unpriv, priv = d["unpriv_test"].to_numpy(), d["priv_test"].to_numpy()
    return dict(
        DP_Diff=dp_diff(y_test, unpriv, priv),
        DIR=dir_ratio(y_test, unpriv, priv),
        Consistency=consistency(X_test, y_test),
    )


def compare(name, d, feats, prep, y_mass):
    rows = {
        "original test labels": true_label_measures(d, feats, prep),
        "baseline (original train labels)": evaluate_test(d, feats, prep, d["y_train"].to_numpy()),
        "massaged (CND)": evaluate_test(d, feats, prep, y_mass),
    }
    table = pd.DataFrame(rows).T.round(4)
    print(f"\n{name}\n{table}")
    return table

In [79]:
table_adult = compare("Adult", data["Adult"], features_adult, prep_adult, adult_y_mass)
table_compas = compare("COMPAS", data["COMPAS"], features_compas, prep_compas, compas_y_mass)

table_german = compare("German", data["German"], features_german, prep_german, german_y_mass)


Adult
                                  DP_Diff     DIR  Consistency  Accuracy
original test labels              -0.1962  0.3583       0.7904       NaN
baseline (original train labels)  -0.1347  0.8190       0.9209    0.5205
massaged (CND)                     0.0056  1.0073       0.9252    0.4373

COMPAS
                                  DP_Diff     DIR  Consistency  Accuracy
original test labels              -0.1210  0.8005       0.5918       NaN
baseline (original train labels)  -0.1400  0.8444       0.9925    0.6088
massaged (CND)                    -0.1321  0.8511       0.9928    0.6124

German
                                  DP_Diff     DIR  Consistency  Accuracy
original test labels              -0.0795  0.8903       0.6632       NaN
baseline (original train labels)  -0.0311  0.6745       0.8748     0.350
massaged (CND)                    -0.0555  0.6638       0.8000     0.392
